# Урок 11 - Протокол за Контекст на Модел (MCP)

**Протоколът за Контекст на Модел (MCP)** е отворен стандарт, който позволява на агентите динамично да откриват и използват инструменти, ресурси и източници на данни по време на изпълнение. Вместо да се кодират инструментите директно в агента, MCP дава възможност на агентите да се свързват с външни сървъри, които предоставят възможности при поискване.

В този урок ще научите:
- Какво е MCP и защо е важно за системите с агенти
- Как работи архитектурата клиент-сървър на MCP
- Как да изградите агенти, които използват откриване на инструменти в стил MCP


## Настройка

**Предварителни изисквания:**
- Проект Microsoft Foundry с разположен модел
- Стартирайте `az login` за удостоверяване


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Какво е Протокол за контекст на модела (MCP)?

MCP определя стандартен начин за AI агенти да откриват и взаимодействат с външни инструменти и източници на данни:

- **MCP сървър**: Предоставя инструменти, ресурси и подсказки чрез стандартен протокол
- **MCP клиент**: Изпълнителната среда на агента, която се свързва със сървърите и открива наличните възможности
- **Динамично откриване**: Агентите не се нуждаят от жорстоко кодирани инструменти — те откриват наличното по време на изпълнение

Това е мощно за изграждане на разширяеми агентски системи, където нови възможности могат да се добавят без промяна на кода на агента.


## Как работи MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Агентът (MCP клиент) се свързва с MCP сървър
2. Сървърът отговаря със списък на наличните инструменти и техните схеми
3. След това агентът може да извиква всеки открит инструмент по време на своето разсъждение
4. Резултатите се връщат обратно по същия протокол


## Симулиране на откриването на MCP инструмент

Тъй като истинският MCP сървър изисква работещ сървърен процес, ще демонстрираме модела с помощта на функции `@tool`, които симулират това, което би предоставила услуга за настаняване, свързана с MCP.

В продукция тези инструменти биха били откривани динамично от MCP сървър, а не дефинирани локално.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Създаване на агент с инструменти в стил MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP в продукция

В производствена среда MCP позволява мощни модели:

- **Динамично откриване на инструменти**: Агентите се свързват с MCP сървъри и откриват инструменти по време на изпълнение
- **Развързана архитектура**: Доставчиците на инструменти могат да се актуализират независимо от агента
- **Споделяне между организации**: Екипите могат да предоставят възможности чрез MCP сървъри, които всеки агент може да използва
- **Поддръжка на Microsoft Agent Framework**: MAF има вградена поддръжка на MCP клиент чрез интеграцията `mcp`

За да използвате реален MCP сървър с MAF, бихте се свързали чрез `hosted_mcp_tool()` или MCP клиентската интеграция.

**Научете повече:**
- [MCP спецификация](https://modelcontextprotocol.io/)
- [Поддръжка на MCP в Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

In this lesson, you learned:
- **MCP** is an open standard for dynamic tool discovery between agents and tool providers
- The **client-server architecture** lets agents discover capabilities at runtime
- MCP enables **extensible, decoupled agent systems** where tools can be added without code changes
- Microsoft Agent Framework provides **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
